In [96]:
import pandas as pd
import numpy as np
import re


def clean_financial_table(df):

    df = df.copy()

    # 1️⃣ Replace blank strings with NaN
    df = df.replace(r'^\s*$', np.nan, regex=True)

    # 2️⃣ Drop completely empty rows and columns
    df = df.dropna(axis=0, how="all")
    df = df.dropna(axis=1, how="all")

    # 3️⃣ Reset index
    df = df.reset_index(drop=True)

    # 4️⃣ Detect header row
    header_row = None

    for i in range(min(3, len(df))):

        row_text = " ".join(df.iloc[i].astype(str).str.lower())

        if re.search(r'(fy|hy|FY|HY|20\d{2}|q[1-4])', row_text):
            header_row = i
            break

    # Promote header if detected
    if header_row is not None:

        df.columns = df.iloc[header_row]
        df = df.drop(header_row).reset_index(drop=True)

    else:
        df.columns = [f"col_{i}" for i in range(len(df.columns))]

    # 5️⃣ Normalize column names
    df.columns = (
        pd.Series(df.columns)
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )

    # 6️⃣ Ensure first column is metric column
    df = df.rename(columns={df.columns[0]: "metric"})

    # 7️⃣ Clean metric labels (remove units)
    def clean_metric(text):

        if pd.isna(text):
            return text

        text = str(text).lower()

        # remove units like ($m)
        text = re.sub(r"\(.*?\)", "", text)

        # remove trailing footnote numbers
        text = re.sub(r"\s+\d+$", "", text)

        # remove superscript footnotes
        text = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰]", "", text)

        # remove star footnotes
        text = re.sub(r"\*", "", text)

        text = text.strip()

        return text

    # 8️⃣ Convert numbers
    # def clean_number(x):

    #     if pd.isna(x):
    #         return x

    #     x = str(x)

    #     # remove commas used as thousands separators
    #     x = x.replace(",", "")

    #     # remove superscript footnotes and stars
    #     x = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰\*]", "", x)

    #     # extract all numeric values
    #     # numbers = re.findall(r"\d+\.?\d*", x)
    #     numbers = re.findall(r"\d+\.?\d*|\(\d+\.?\d*\)", x)

    #     if numbers:
    #         value = float(numbers[0])

    #         # handle percentages
    #         if "%" in x:
    #             return value / 100

    #         return value

    #     return x

    def clean_number(x):

        if pd.isna(x):
            return x

        x = str(x).strip()

        # detect negative via parentheses
        is_negative = bool(re.search(r"\(\s*\d+\.?\d*", x))

        # remove commas
        x = x.replace(",", "")

        # remove footnote markers
        x = re.sub(r"[¹²³⁴⁵⁶⁷⁸⁹⁰\*]", "", x)

        # extract all numbers
        numbers = re.findall(r"\d+\.?\d*", x)

        if numbers:
            value = float(numbers[0])

            if is_negative:
                value = -value

            # percentage handling
            if "%" in x:
                return value / 100

            return value

        return x 
   

    #Cleans only for metric column.
    df['metric'] = df['metric'].map(clean_metric)

    #Apply clean_number_conditionally only to non metric cols. 
    cols_excluding_metric = df.columns.difference(['metric'])
    df[cols_excluding_metric] = df[cols_excluding_metric].map(clean_number)


    return df

In [97]:
test_table = r"datasets\processed\tables\ABG\FY17\ABG_FY17_IP-table-1"

df = pd.read_csv(test_table,index_col=0)

In [98]:
df

,0,1,2,3,4,5
0,NaN,Commercial,Storage,Development,FY17 Total,FY16 Total
1,Rental income,75.3,69.7,NaN,145.0,131.3
2,Finance income 1,NaN,NaN,43.9,43.9,46.3
3,Fee income 2,12.6,NaN,NaN,12.6,12.5
4,Share of profit from equity accounted,"32.8 4, 6",NaN,20.7,53.5,19.4
5,Sale of inventory,NaN,NaN,NaN,0.0,2.9
6,Net change in fair value of investments,43.5,NaN,NaN,43.5,16.0
7,Other Income,2.4,NaN,1.1,3.5,0.0
8,Interest,NaN,NaN,NaN,0.4,0.5
9,Total Underlying Revenue,166.6,69.7,65.7,302.4,228.9


In [99]:
clean_financial_table(df=df)

,metric,commercial,storage,development,fy17_total,fy16_total
0,rental income,75.3,69.7,NaN,145.0,131.3
1,finance income,NaN,NaN,43.9,43.9,46.3
2,fee income,12.6,NaN,NaN,12.6,12.5
3,share of profit from equity accounted,32.8,NaN,20.7,53.5,19.4
4,sale of inventory,NaN,NaN,NaN,0.0,2.9
5,net change in fair value of investments,43.5,NaN,NaN,43.5,16.0
6,other income,2.4,NaN,1.1,3.5,0.0
7,interest,NaN,NaN,NaN,0.4,0.5
8,total underlying revenue,166.6,69.7,65.7,302.4,228.9
9,expenses,-15.2,-25.9,-3.0,-44.1,-42.8
